# exp11: Tahap pertama berbasis kalender pada PharmaSales

Notebook **pelapor**: membaca berkas hasil yang sudah ada, tidak melatih model,
tidak menulis ke `results/`, tidak mengimpor `src/experiments` maupun `tools`.

## 1. Ringkasan

**Pertanyaan yang dijawab.** RQ1, celah yang ditinggalkan naskah IJIES: tahap pertama
struktural yang membalikkan hasil pada data ritel membutuhkan kunci kategorikal seperti toko,
hari dan promosi, sedangkan deret obat tidak memiliki satu pun di antaranya. Dengan membentuk
kunci kalender berupa hari dalam minggu dan bulan, tahap pertama struktural menjadi dapat
dipakai. Pertanyaannya, apakah penyelarasan struktur kemudian memperbaiki akurasi pada deret
obat seperti pada Rossmann, atau kondisi residualnya tetap *noise*.

Eksperimen ini menggabungkan dua temuan sebelumnya, yaitu Subbab 4.2.3 tentang desain sinyal
masukan yang lebih menentukan daripada arsitektur, dan Subbab 4.4.2 tentang tahap pertama yang
sesuai struktur data sebagai penentu keberhasilan hibridisasi residual.

**Ramalan tercatat di dalam *docstring* skripnya sendiri**, bukan sebagai berkas
pra-registrasi terpisah seperti exp10, exp13 dan exp14. Skrip dan berkas hasilnya masuk pada
commit yang sama, yaitu `e4c061d`. Oleh karena itu, urutan waktu antara penulisan ramalan
dan diperolehnya hasil tidak terdokumentasi.

**Dipakai di mana.**

| Naskah | Bagian |
|---|---|
| Disertasi | Subbab 4.6.3, **Tabel 4.12** |
| Paper IJIES | **tidak dipakai** |

> **Penting:** exp11 mengubah **set fitur**, sehingga hasilnya dilaporkan sebagai perluasan
> berlabel, **bukan pengganti** exp05c. Acuan pembandingnya tetap S₁ linier exp05c.

## 2. Lingkungan

In [1]:
# Notebook pelapor: HANYA MEMBACA.
# Tidak melatih model, tidak mengimpor src/experiments atau tools (agar TensorFlow
# tidak ikut termuat), dan tidak pernah menulis ke results/, data/ atau raw/.
%matplotlib inline
from pathlib import Path
import ast, json, hashlib, datetime, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def cari_root(p: Path) -> Path:
    for c in [p, *p.parents]:
        if (c / "results").is_dir() and (c / "src" / "experiments").is_dir():
            return c
    raise RuntimeError("root repositori tidak ditemukan dari " + str(p))


ROOT = cari_root(Path.cwd().resolve())
RES = ROOT / "results"

pd.set_option("display.width", 210)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 95)
pd.set_option("display.max_rows", 200)

print("Root repositori :", ROOT)
print("Folder hasil    :", RES.relative_to(ROOT),
      f"({sum(1 for f in RES.rglob('*') if f.is_file())} berkas)")
print("Dijalankan      :", datetime.datetime.now().strftime("%Y-%m-%d %H:%M"))


def idn(x, d=2, tanda=False):
    """Format angka gaya Indonesia (koma desimal, titik ribuan)."""
    s = f"{x:,.{d}f}".replace(",", "X").replace(".", ",").replace("X", ".")
    if tanda and x > 0:
        s = "+" + s
    return s.replace("-", "\u2212")


def baca(nama, **kw):
    """Baca satu berkas di results/ sebagai DataFrame. Read-only."""
    f = RES / nama
    if not f.exists():
        raise FileNotFoundError(f"{nama} tidak ada di results/")
    return pd.read_csv(f, **kw)


def potongan(relpath, nama=None, awal=None, akhir=None):
    """Tampilkan potongan kode dengan membaca berkas sebagai TEKS.

    Memakai ``ast`` untuk menemukan batas fungsi/konstanta, sehingga modul
    eksperimen tidak perlu diimpor.
    """
    f = ROOT / relpath
    teks = f.read_text(encoding="utf-8")
    baris = teks.splitlines()
    if nama is not None:
        pohon = ast.parse(teks)
        ketemu = False
        for n in ast.walk(pohon):
            if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)) and n.name == nama:
                awal, akhir, ketemu = n.lineno, n.end_lineno, True
                break
            if isinstance(n, ast.Assign) and any(getattr(t, "id", None) == nama for t in n.targets):
                awal, akhir, ketemu = n.lineno, n.end_lineno, True
                break
        if not ketemu:
            raise KeyError(f"'{nama}' tidak ditemukan di {relpath}")
    awal = awal or 1
    akhir = akhir or len(baris)
    lebar = len(str(akhir))
    print(f"--- {relpath}:{awal}-{akhir} " + "-" * max(0, 74 - len(relpath)))
    for i in range(awal, akhir + 1):
        print(f"{str(i).rjust(lebar)} | {baris[i - 1]}")


def git(*args):
    """Jalankan perintah git read-only di root repositori."""
    try:
        r = subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, timeout=30)
        return r.stdout.strip() if r.returncode == 0 else ""
    except Exception:
        return ""


def asal(nama_meta=None, berkas_hasil=None, skrip=None, log=None):
    """Cetak asal-usul: meta.json, commit, dan log run bila ada."""
    if nama_meta:
        f = RES / nama_meta
        if f.exists():
            print("meta.json " + "-" * 60)
            print(json.dumps(json.loads(f.read_text(encoding="utf-8")), indent=2, ensure_ascii=False))
            print()
    for label, path in [("berkas hasil", berkas_hasil), ("skrip", skrip)]:
        if path:
            print(f"commit terakhir yang menyentuh {label} ({path})")
            print("  " + (git("log", "-1", "--date=short", "--format=%h  %ad  %s", "--", path) or "(tidak terbaca)"))
    if log:
        f = RES / "logs" / log
        print()
        if f.exists():
            print(f"results/logs/{log}  ({f.stat().st_size} bytes, "
                  f"ditulis {datetime.datetime.fromtimestamp(f.stat().st_mtime):%Y-%m-%d %H:%M})")
        else:
            print(f"results/logs/{log} tidak ada di klon ini (berkas *.log diabaikan git).")

Root repositori : I:\Shared drives\Riset S3\Experiment S3 Code\antgravity_riset\AR-LRX_hybrid-forecasting


Folder hasil    : results (179 berkas)
Dijalankan      : 2026-09-29 19:30


In [2]:
# Pembanding angka CSV terhadap angka yang tercetak di disertasi.
# Tidak ada angka yang dibetulkan di sini; ketidakcocokan hanya dilaporkan.
_cek = []


def cek(butir, nilai_csv, nilai_naskah, tol=0.005, d=4, rujukan=""):
    beda = abs(float(nilai_csv) - float(nilai_naskah))
    ok = beda <= tol
    _cek.append({
        "Butir": butir,
        "Rujukan naskah": rujukan,
        "Dari CSV": round(float(nilai_csv), d),
        "Di disertasi": float(nilai_naskah),
        "|selisih|": round(beda, 6),
        "Status": "COCOK" if ok else "TIDAK COCOK",
    })
    return ok


def laporan_cek():
    t = pd.DataFrame(_cek)
    n_ok = int((t.Status == "COCOK").sum())
    display(t)
    print()
    print(f"RINGKASAN: {n_ok} dari {len(t)} angka COCOK dengan naskah disertasi.")
    buruk = t[t.Status == "TIDAK COCOK"]
    if len(buruk):
        print()
        print("!" * 78)
        print("ADA ANGKA YANG TIDAK COCOK. Tidak dibetulkan di notebook ini.")
        print("Laporkan butir berikut sebelum notebook dipakai sebagai lampiran:")
        for _, r in buruk.iterrows():
            print(f"  - {r.Butir} ({r['Rujukan naskah']}): "
                  f"CSV {r['Dari CSV']} vs naskah {r['Di disertasi']}")
        print("!" * 78)
    else:
        print("Tidak ada selisih di luar toleransi pembulatan.")
    return t

## 3. Desain eksperimen

**Kunci kalender yang ditambahkan**, dalam urutan hierarki:

In [3]:
potongan("tools/exp11_pharma_calendar_s1.py", nama="KEY_COLS")

--- tools/exp11_pharma_calendar_s1.py:88-88 -----------------------------------------
88 | KEY_COLS = ("DayOfWeek", "Month")          # structural keys, in hierarchy order


**Cara kunci itu dibentuk**, dibaca sebagai teks tanpa mengimpor modulnya:

In [4]:
potongan("tools/exp11_pharma_calendar_s1.py", nama="add_calendar")

--- tools/exp11_pharma_calendar_s1.py:95-117 -----------------------------------------
 95 | def add_calendar(d: P.Dataset, granularity: str) -> P.Dataset:
 96 |     """Return a copy of `d` with calendar columns appended to the feature list.
 97 |     Rows and split points are untouched (no NaN is introduced)."""
 98 |     f = d.frame.copy()
 99 |     ds = pd.to_datetime(f["ds"])
100 |     new = []
101 |     f["Month"] = ds.dt.month.astype(float)
102 |     if granularity == "daily":
103 |         f["DayOfWeek"] = ds.dt.dayofweek.astype(float)
104 |         for k in range(1, 7):
105 |             f[f"dow_{k}"] = (ds.dt.dayofweek == k).astype(float)
106 |             new.append(f"dow_{k}")
107 |         keys = ["DayOfWeek", "Month"]
108 |     else:
109 |         keys = ["Month"]
110 |     for m in range(2, 13):
111 |         f[f"month_{m}"] = (ds.dt.month == m).astype(float)
112 |         new.append(f"month_{m}")
113 |     return P.Dataset(name=d.name, feature_set=d.feature_set + "+cal",

**Tahap pertama yang dibandingkan.**

| Label | Isi |
|---|---|
| `S1 [linear+cal]` | regresi linier atas lag **ditambah** peubah boneka kalender |
| `S1 [structural]` | rata-rata per kelompok kalender saja, tanpa lag |
| `S1 [struct_linear]` | struktural digabung linier |
| `XGBoost [+cal]` | pembanding yang diberi fitur kalender yang sama, agar adil |

**Kontrak C1–C8 tidak diubah.** Acuan seluruh perbandingan adalah `S1 [linear] (exp05c)`,
yaitu tahap pertama linier tanpa kalender, sehingga efek penambahan kalender terbaca langsung.

## 4. Asal-usul hasil

In [5]:
RUN_EXPERIMENT = False

PERINTAH = [
    "python tools/exp11_pharma_calendar_s1.py",
]

if RUN_EXPERIMENT:
    for c in PERINTAH:
        print(">>", c)
        r = subprocess.run(c, shell=True, cwd=ROOT)
        print("   kode keluar:", r.returncode)
else:
    print("RUN_EXPERIMENT = False - eksperimen TIDAK dijalankan ulang.")
    print()
    print("Perintah asli yang menghasilkan berkas di results/:")
    for c in PERINTAH:
        print("   ", c)
    print()
    print('Dijalankan 25 September 2026, sebelum exp12-exp14.')

RUN_EXPERIMENT = False - eksperimen TIDAK dijalankan ulang.

Perintah asli yang menghasilkan berkas di results/:
    python tools/exp11_pharma_calendar_s1.py

Dijalankan 25 September 2026, sebelum exp12-exp14.


In [6]:
asal(nama_meta="exp11_pharma_calendar.meta.json", berkas_hasil="results/exp11_pharma_calendar.csv", skrip="tools/exp11_pharma_calendar_s1.py")

meta.json ------------------------------------------------------------
{
  "experiment": "exp11_pharma_calendar",
  "n_rows": 288,
  "protocol": {
    "split_ratios": [
      0.7,
      0.15,
      0.15
    ],
    "tuning": "validation-only, refit on train+val",
    "feature_sets": [
      "A_lag1",
      "B_rich"
    ]
  },
  "environment": {
    "python": "3.10.11",
    "platform": "Windows-10-10.0.22621-SP0",
    "numpy": "2.2.6",
    "pandas": "2.3.3",
    "seed": "42",
    "sklearn": "1.7.2",
    "xgboost": "3.2.0",
    "statsmodels": "0.14.6"
  }
}

commit terakhir yang menyentuh berkas hasil (results/exp11_pharma_calendar.csv)


  e4c061d  2026-09-25  Add exp11 (calendar-keyed first stage on PharmaSales) and ICoICT leakage audit
commit terakhir yang menyentuh skrip (tools/exp11_pharma_calendar_s1.py)


  e4c061d  2026-09-25  Add exp11 (calendar-keyed first stage on PharmaSales) and ICoICT leakage audit


## 5. Hasil

### 5.1 Tabel 4.12

In [7]:
e11 = baca("exp11_pharma_calendar_summary.csv")
d11 = e11[(e11.scope == "distinct") & (e11.reference == "S1 [linear] (exp05c)")].set_index(
    ["model", "granularity"])

LAB = [("S1 [linear+cal]", "S1 linier + kalender"),
       ("AR-LRX [linear+cal]", "AR-LRX, S1 linier + kalender"),
       ("AR-LRX [struct_linear]", "AR-LRX, S1 struktural-linier"),
       ("AR-LRX-Aug [struct_linear]", "AR-LRX augmented, S1 struktural-linier"),
       ("S1 [structural]", "S1 struktural saja"),
       ("XGBoost [+cal]", "XGBoost + kalender")]
rows = []
for k, l in LAB:
    a, b = d11.loc[(k, "daily")], d11.loc[(k, "weekly")]
    rows.append([l, f"{int(a.wins)}/10", int(a.sig_wins_bh), round(a.pooled_pct, 2),
                 f"{int(b.wins)}/10", int(b.sig_wins_bh), round(b.pooled_pct, 2)])
t412 = pd.DataFrame(rows, columns=["Model", "Harian: menang", "Harian: signifikan",
                                   "Harian: gabungan (%)", "Mingguan: menang",
                                   "Mingguan: signifikan", "Mingguan: gabungan (%)"])
display(t412)
print()
print("Acuan = S1 linier exp05c (setara regresi linier, dan setara AR-LRX exp05c).")
print("Gabungan negatif berarti lebih akurat daripada acuan.")

,Model,Harian: menang,Harian: signifikan,Harian: gabungan (%),Mingguan: menang,Mingguan: signifikan,Mingguan: gabungan (%)
0,S1 linier + kalender,9/10,2,-2.44,4/10,0,0.69
1,"AR-LRX, S1 linier + kalender",9/10,2,-2.35,4/10,0,0.95
2,"AR-LRX, S1 struktural-linier",7/10,1,-1.43,4/10,0,0.95
3,"AR-LRX augmented, S1 struktural-linier",9/10,1,-1.99,4/10,0,0.26
4,S1 struktural saja,6/10,0,-0.36,2/10,0,9.83
5,XGBoost + kalender,8/10,0,-1.88,3/10,0,3.63



Acuan = S1 linier exp05c (setara regresi linier, dan setara AR-LRX exp05c).
Gabungan negatif berarti lebih akurat daripada acuan.


### 5.2 Asal perbaikannya: tahap pertama atau koreksi residual?

Pertanyaan tersebut menjadi inti eksperimen ini.

In [8]:
raw11 = baca("exp11_pharma_calendar.csv")
ar = raw11[raw11.model == "AR-LRX [linear+cal]"]
print(f"Konfigurasi AR-LRX [linear+cal] : {len(ar)}")
print(f"Gerbang menutup (w* = 0)        : {int((ar.gate_w == 0).sum())} dari {len(ar)}")
print(f"R2 residual validasi > 0,05     : {int((ar.resid_val_r2 > 0.05).sum())} dari {len(ar)}")
print(f"R2 residual validasi maksimum   : {ar.resid_val_r2.max():.4f}")
print()
sn = e11[(e11.scope == "distinct") & (e11.model == "AR-LRX [linear+cal]")
         & (e11.reference == "S1 [linear+cal]")]
print("AR-LRX [linear+cal] terhadap tahap pertama BERKALENDERNYA SENDIRI:")
print(sn[["granularity", "wins", "sig_wins_bh", "sig_losses_bh", "pooled_pct"]].round(3).to_string(index=False))
print()
xg = e11[(e11.scope == "distinct") & (e11.model == "AR-LRX [linear+cal]")
         & (e11.reference == "XGBoost [+cal]")]
if len(xg):
    print("AR-LRX [linear+cal] terhadap XGBoost yang juga diberi fitur kalender:")
    print(xg[["granularity", "wins", "pooled_pct"]].round(3).to_string(index=False))
else:
    print("Acuan pembanding yang tersedia di berkas ringkasan:")
    print(sorted(e11.reference.unique()))

Konfigurasi AR-LRX [linear+cal] : 32
Gerbang menutup (w* = 0)        : 10 dari 32
R2 residual validasi > 0,05     : 0 dari 32
R2 residual validasi maksimum   : 0.0069

AR-LRX [linear+cal] terhadap tahap pertama BERKALENDERNYA SENDIRI:
granularity  wins  sig_wins_bh  sig_losses_bh  pooled_pct
      daily     3            0              0       0.099
     weekly     4            0              0       0.253

Acuan pembanding yang tersedia di berkas ringkasan:
['S1 [linear+cal]', 'S1 [linear] (exp05c)']


## 6. Cek kecocokan dengan angka disertasi

In [9]:
cek("Tabel 4.12: S1 linier+kal harian, gabungan (%)", d11.loc[("S1 [linear+cal]", "daily")].pooled_pct,
    -2.44, tol=0.005, rujukan="Tabel 4.12 / Subbab 4.6.3")
cek("Tabel 4.12: S1 linier+kal harian, menang", d11.loc[("S1 [linear+cal]", "daily")].wins, 9,
    tol=0, rujukan="Tabel 4.12 / Subbab 4.6.3")
cek("Tabel 4.12: S1 linier+kal harian, signifikan", d11.loc[("S1 [linear+cal]", "daily")].sig_wins_bh, 2,
    tol=0, rujukan="Subbab 4.6.3")
cek("Tabel 4.12: AR-LRX linier+kal harian, gabungan (%)",
    d11.loc[("AR-LRX [linear+cal]", "daily")].pooled_pct, -2.35, tol=0.005, rujukan="Subbab 4.6.3")
cek("Tabel 4.12: AR-LRX linier+kal harian, menang", d11.loc[("AR-LRX [linear+cal]", "daily")].wins, 9,
    tol=0, rujukan="Subbab 4.6.3")
cek("Tabel 4.12: S1 linier+kal mingguan, gabungan (%)",
    d11.loc[("S1 [linear+cal]", "weekly")].pooled_pct, 0.69, tol=0.005, rujukan="Subbab 4.6.3")
cek("Tabel 4.12: AR-LRX linier+kal mingguan, gabungan (%)",
    d11.loc[("AR-LRX [linear+cal]", "weekly")].pooled_pct, 0.95, tol=0.005, rujukan="Subbab 4.6.3")
cek("Tabel 4.12: S1 struktural mingguan, gabungan (%)",
    d11.loc[("S1 [structural]", "weekly")].pooled_pct, 9.83, tol=0.005, rujukan="Subbab 4.6.3")
cek("Subbab 4.6.3: AR-LRX vs tahap pertama berkalendernya sendiri (%)",
    sn[sn.granularity == "daily"].pooled_pct.iloc[0], 0.10, tol=0.005, rujukan="Subbab 4.6.3")
cek("Subbab 4.6.3: konfigurasi dengan R2 residual validasi > 0,05",
    int((ar.resid_val_r2 > 0.05).sum()), 0, tol=0, rujukan="Subbab 4.6.3")
cek("Subbab 4.6.3: gerbang menutup (dari 32)", int((ar.gate_w == 0).sum()), 10, tol=0,
    rujukan="Subbab 4.6.3")

hasil_cek = laporan_cek()

,Butir,Rujukan naskah,Dari CSV,Di disertasi,|selisih|,Status
0,"Tabel 4.12: S1 linier+kal harian, gabungan (%)",Tabel 4.12 / Subbab 4.6.3,-2.4431,-2.44,0.003106,COCOK
1,"Tabel 4.12: S1 linier+kal harian, menang",Tabel 4.12 / Subbab 4.6.3,9.0000,9.00,0.000000,COCOK
2,"Tabel 4.12: S1 linier+kal harian, signifikan",Subbab 4.6.3,2.0000,2.00,0.000000,COCOK
3,"Tabel 4.12: AR-LRX linier+kal harian, gabungan (%)",Subbab 4.6.3,-2.3463,-2.35,0.003721,COCOK
4,"Tabel 4.12: AR-LRX linier+kal harian, menang",Subbab 4.6.3,9.0000,9.00,0.000000,COCOK
5,"Tabel 4.12: S1 linier+kal mingguan, gabungan (%)",Subbab 4.6.3,0.6935,0.69,0.003533,COCOK
6,"Tabel 4.12: AR-LRX linier+kal mingguan, gabungan (%)",Subbab 4.6.3,0.9481,0.95,0.001891,COCOK
7,"Tabel 4.12: S1 struktural mingguan, gabungan (%)",Subbab 4.6.3,9.8308,9.83,0.000806,COCOK
8,Subbab 4.6.3: AR-LRX vs tahap pertama berkalendernya sendiri (%),Subbab 4.6.3,0.0993,0.10,0.000748,COCOK
9,"Subbab 4.6.3: konfigurasi dengan R2 residual validasi > 0,05",Subbab 4.6.3,0.0000,0.00,0.000000,COCOK



RINGKASAN: 11 dari 11 angka COCOK dengan naskah disertasi.
Tidak ada selisih di luar toleransi pembulatan.


## 7. Interpretasi dan keterbatasan

**Pada data harian, fitur kalender memberi perbaikan akurasi pertama yang sah pada data
obat.** Tahap pertama linier dengan kalender menurunkan RMSE gabungan 2,44% dan menang pada
9 dari 10 konfigurasi harian, 2 di antaranya signifikan setelah penyesuaian
Benjamini–Hochberg. AR-LRX dengan tahap pertama tersebut menurunkan RMSE 2,35% dengan pola
yang sama.

**Pada data mingguan tidak ada perbaikan** (+0,69% dan +0,95%), karena hari dalam minggu
tidak lagi bermakna setelah agregasi.

**Perbaikan itu datang dari tahap pertama, bukan dari koreksi residual.** Tiga bukti yang
sejalan: terhadap tahap pertama berkalendernya sendiri AR-LRX hanya berselisih +0,10%;
tidak ada satu pun konfigurasi dengan R² residual validasi di atas 0,05; dan gerbang menutup
pada 10 dari 32 konfigurasi. Kondisi residual data obat **tetap** kondisi *noise* walaupun
tahap pertamanya diperbaiki.

**Penaksir struktural murni lebih buruk pada data mingguan**, yaitu +9,83%, karena rata-rata
per bulan dari 4–5 tahun data terlalu kasar untuk deret mingguan.

**Keterbatasan.**

1. exp11 mengubah set fitur, sehingga hasilnya tidak sebanding langsung dengan exp05c dan
   hanya dapat dibaca sebagai perluasan berlabel.
2. Kunci kalender terbatas pada hari dalam minggu dan bulan. Hari libur, akhir bulan, dan
   musim penyakit tidak diikutkan.
3. Perbaikan 2,35% masih belum menutup selisih terhadap ETS musiman pada data harian
   (exp14): secara eksploratif masih tertinggal 2,27%.
4. Ramalannya tercatat di *docstring* skrip, bukan sebagai berkas pra-registrasi terpisah
   seperti exp10, exp13 dan exp14. Skrip dan hasilnya masuk pada commit yang sama (`e4c061d`),
   sehingga urutan waktu antara ramalan dan hasil tidak terdokumentasi.

## 8. Pertanyaan yang mungkin diajukan promotor

**1. "Kalau kalender memperbaiki akurasi, mengapa tidak dipakai sejak awal di exp05c?"**
Karena exp05c dirancang untuk menguji AR-LRX pada kondisi paling miskin informasi, yaitu deret
univariat dengan tahap kedua yang hanya memiliki fitur lag. Kondisi itulah yang perlu diuji
untuk memperlihatkan sifat pengaman gerbang. exp11 merupakan langkah berikutnya, dan
dilaporkan sebagai perluasan berlabel.

**2. "Apakah perbaikan 2,35% itu datang dari AR-LRX atau dari kalendernya?"**
Dari kalendernya. Tahap pertama linier berkalender sendiri sudah memberi −2,44%; AR-LRX di
atasnya hanya berselisih +0,10% terhadap tahap pertama itu. Bagian 5.2 notebook ini
menunjukkan angkanya, bersama dua bukti pendukung: tidak ada R² residual validasi di atas
0,05, dan gerbang menutup pada 10 dari 32 konfigurasi.

**3. "Apakah pembandingnya adil, dalam arti XGBoost juga diberi fitur kalender?"**
Ya. Baris `XGBoost [+cal]` pada Tabel 4.12 adalah XGBoost dengan fitur kalender yang sama.
Tanpa itu, perbandingannya akan menguntungkan AR-LRX secara artifisial.

**4. "Mengapa pada data mingguan hasilnya memburuk?"**
Hari dalam minggu kehilangan maknanya setelah agregasi mingguan, sehingga yang tersisa hanya
kunci bulan. Rata-rata per bulan dari 4–5 tahun data terlalu kasar untuk deret mingguan. Hal
ini paling terlihat pada `S1 [structural]` yang memburuk sebesar +9,83%.

**5. "Apakah ini membatalkan kesimpulan bahwa data obat berada pada kondisi residual *noise*?"**
Tidak. Setelah tahap pertamanya diperbaiki dengan kalender, residualnya tetap tidak dapat
dipelajari, karena R² residual validasi tidak pernah melampaui 0,05. Yang berubah adalah
akurasi tahap pertama, sedangkan kelayakan koreksi residual tetap sama.